# 任务二：Unet-5模型性能分析

## 实验目的
通过调节实验参数，分析不同参数设置对Unet模型性能的影响。

## 实验参数
- 训练数据比例
- 批次大小
- 迭代轮数
- 学习速率

## 实验设计
本实验将对比以下参数设置：
1. **基准实验**：原始参数设置
   - 学习率: 0.0001
   - 批次大小: 16
   - 迭代轮数: 2（为演示目的减少，实际应为400）

2. **对比实验1**：改变学习率
   - 学习率: 0.001（提高10倍）
   - 批次大小: 16
   - 迭代轮数: 2

3. **对比实验2**：改变批次大小
   - 学习率: 0.0001
   - 批次大小: 8（减小）
   - 迭代轮数: 2

In [ ]:
# 导入实验所需要的库
import os
import argparse
import ast
import numpy as np
import cv2
import mindspore
import mindspore.nn as nn
import mindspore.ops.operations as F
from mindspore import Model, context
from mindspore.nn.loss.loss import _Loss
from mindspore.communication.management import init, get_group_size
from mindspore.train.callback import CheckpointConfig, ModelCheckpoint
from mindspore.context import ParallelMode
from mindspore.train.serialization import load_checkpoint, load_param_into_net
from mindspore.common.initializer import TruncatedNormal
from mindspore.nn import CentralCrop
from PIL import Image, ImageSequence
import mindspore.dataset as ds
import mindspore.dataset.vision.c_transforms as c_vision
from mindspore.dataset.vision.utils import Inter
from mindspore.communication.management import get_rank, get_group_size
from collections import deque
import time
from mindspore.train.callback import Callback
from mindspore.common.tensor import Tensor
from scipy.special import softmax
from matplotlib import pyplot as plt

# 设置设备
device_id = 2
context.set_context(mode=context.GRAPH_MODE, device_target="Ascend", save_graphs=False)
mindspore.set_seed(1)

print("库导入完成！")

## 一、定义Unet网络结构

In [ ]:
class DoubleConv(nn.Cell):
    """定义两个卷积层"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        init_value_0 = TruncatedNormal(0.06)
        init_value_1 = TruncatedNormal(0.06)
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.SequentialCell(
            [nn.Conv2d(in_channels, mid_channels, kernel_size=3, has_bias=True,
                       weight_init=init_value_0, pad_mode="valid"),
             nn.ReLU(),
             nn.Conv2d(mid_channels, out_channels, kernel_size=3, has_bias=True,
                       weight_init=init_value_1, pad_mode="valid"),
             nn.ReLU()]
        )

    def construct(self, x):
        return self.double_conv(x)


class Down(nn.Cell):
    """下采样模块：最大池化接两个卷积层"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.SequentialCell(
            [nn.MaxPool2d(kernel_size=2, stride=2),
             DoubleConv(in_channels, out_channels)]
        )

    def construct(self, x):
        return self.maxpool_conv(x)


class Up1(nn.Cell):
    """第一个上采样模块"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.concat = F.Concat(axis=1)
        self.factor = 56.0 / 64.0
        self.center_crop = CentralCrop(central_fraction=self.factor)
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        self.up = nn.Conv2dTranspose(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.relu = nn.ReLU()

    def construct(self, x1, x2):
        x1 = self.up(x1)
        x1 = self.relu(x1)
        x2 = self.center_crop(x2)
        x = self.concat((x1, x2))
        return self.conv(x)


class Up2(nn.Cell):
    """第二个上采样模块"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.concat = F.Concat(axis=1)
        self.factor = 104.0 / 136.0
        self.center_crop = CentralCrop(central_fraction=self.factor)
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        self.up = nn.Conv2dTranspose(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.relu = nn.ReLU()

    def construct(self, x1, x2):
        x1 = self.up(x1)
        x1 = self.relu(x1)
        x2 = self.center_crop(x2)
        x = self.concat((x1, x2))
        return self.conv(x)


class Up3(nn.Cell):
    """第三个上采样模块"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.concat = F.Concat(axis=1)
        self.factor = 200 / 280
        self.center_crop = CentralCrop(central_fraction=self.factor)
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        self.up = nn.Conv2dTranspose(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.relu = nn.ReLU()

    def construct(self, x1, x2):
        x1 = self.up(x1)
        x1 = self.relu(x1)
        x2 = self.center_crop(x2)
        x = self.concat((x1, x2))
        return self.conv(x)


class Up4(nn.Cell):
    """第四个上采样模块"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.concat = F.Concat(axis=1)
        self.factor = 392 / 568
        self.center_crop = CentralCrop(central_fraction=self.factor)
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        self.up = nn.Conv2dTranspose(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.relu = nn.ReLU()

    def construct(self, x1, x2):
        x1 = self.up(x1)
        x1 = self.relu(x1)
        x2 = self.center_crop(x2)
        x = self.concat((x1, x2))
        return self.conv(x)


class OutConv(nn.Cell):
    """输出层"""
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        init_value = TruncatedNormal(0.06)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, has_bias=True, weight_init=init_value)

    def construct(self, x):
        x = self.conv(x)
        return x


class UNet(nn.Cell):
    """U-Net网络"""
    def __init__(self, n_channels, n_classes):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        self.up1 = Up1(1024, 512)
        self.up2 = Up2(512, 256)
        self.up3 = Up3(256, 128)
        self.up4 = Up4(128, 64)
        self.outc = OutConv(64, n_classes)

    def construct(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

print("U-Net网络结构定义完成！")

## 二、定义损失函数和评估指标

In [ ]:
class CrossEntropyWithLogits(_Loss):
    """交叉熵损失函数"""
    def __init__(self):
        super(CrossEntropyWithLogits, self).__init__()
        self.transpose_fn = F.Transpose()
        self.reshape_fn = F.Reshape()
        self.softmax_cross_entropy_loss = nn.SoftmaxCrossEntropyWithLogits()
        self.cast = F.Cast()

    def construct(self, logits, label):
        logits = self.transpose_fn(logits, (0, 2, 3, 1))
        logits = self.cast(logits, mindspore.float32)
        label = self.transpose_fn(label, (0, 2, 3, 1))
        loss = self.reduce_mean(self.softmax_cross_entropy_loss(self.reshape_fn(logits, (-1, 2)),
                                                                self.reshape_fn(label, (-1, 2))))
        return self.get_loss(loss)


class dice_coeff(nn.Metric):
    """Dice系数评估指标"""
    def __init__(self):
        super(dice_coeff, self).__init__()
        self.clear()

    def clear(self):
        self._dice_coeff_sum = 0
        self._samples_num = 0

    def update(self, *inputs):
        if len(inputs) != 2:
            raise ValueError('Dice coefficient need 2 inputs (y_pred, y), but got {}'.format(len(inputs)))
        y_pred = self._convert_data(inputs[0])
        y = self._convert_data(inputs[1])
        self._samples_num += y.shape[0]
        y_pred = y_pred.transpose(0, 2, 3, 1)
        y = y.transpose(0, 2, 3, 1)
        y_pred = softmax(y_pred, axis=3)
        inter = np.dot(y_pred.flatten(), y.flatten())
        union = np.dot(y_pred.flatten(), y_pred.flatten()) + np.dot(y.flatten(), y.flatten())
        single_dice_coeff = 2 * float(inter) / float(union + 1e-6)
        self._dice_coeff_sum += single_dice_coeff

    def eval(self):
        if self._samples_num == 0:
            raise RuntimeError('Total samples num must not be 0.')
        return self._dice_coeff_sum / float(self._samples_num)

print("损失函数和评估指标定义完成！")

## 三、定义数据处理函数

In [ ]:
def _load_multipage_tiff(path):
    """加载多页TIFF图像"""
    return np.array([np.array(p) for p in ImageSequence.Iterator(Image.open(path))])


def _get_val_train_indices(length, fold, ratio=0.8):
    """将训练数据分为训练集和验证集"""
    assert 0 < ratio <= 1, "Train/total data ratio must be in range (0.0, 1.0]"
    np.random.seed(0)
    indices = np.arange(0, length, 1, dtype=np.int)
    np.random.shuffle(indices)

    if fold is not None:
        indices = deque(indices)
        indices.rotate(fold * round((1.0 - ratio) * length))
        indices = np.array(indices)
        train_indices = indices[:round(ratio * len(indices))]
        val_indices = indices[round(ratio * len(indices)):]
    else:
        train_indices = indices
        val_indices = []
    return train_indices, val_indices


def data_post_process(img, mask):
    """数据后处理"""
    img = np.expand_dims(img, axis=0)
    mask = (mask > 0.5).astype(np.int)
    mask = (np.arange(mask.max() + 1) == mask[..., None]).astype(int)
    mask = mask.transpose(2, 0, 1).astype(np.float32)
    return img, mask


def train_data_augmentation(img, mask):
    """数据增强"""
    h_flip = np.random.random()
    if h_flip > 0.5:
        img = np.flipud(img)
        mask = np.flipud(mask)
    v_flip = np.random.random()
    if v_flip > 0.5:
        img = np.fliplr(img)
        mask = np.fliplr(mask)
    left = int(np.random.uniform()*0.3*572)
    right = int((1-np.random.uniform()*0.3)*572)
    top = int(np.random.uniform()*0.3*572)
    bottom = int((1-np.random.uniform()*0.3)*572)
    img = img[top:bottom, left:right]
    mask = mask[top:bottom, left:right]
    brightness = np.random.uniform(-0.2, 0.2)
    img = np.float32(img+brightness*np.ones(img.shape))
    img = np.clip(img, -1.0, 1.0)
    return img, mask


def create_dataset(data_dir, repeat=2, train_batch_size=16, augment=False, cross_val_ind=1, run_distribute=False):
    """创建数据集"""
    images = _load_multipage_tiff(os.path.join(data_dir, 'train-volume.tif'))
    masks = _load_multipage_tiff(os.path.join(data_dir, 'train-labels.tif'))

    train_indices, val_indices = _get_val_train_indices(len(images), cross_val_ind)
    train_images = images[train_indices]
    train_masks = masks[train_indices]
    train_images = np.repeat(train_images, repeat, axis=0)
    train_masks = np.repeat(train_masks, repeat, axis=0)
    val_images = images[val_indices]
    val_masks = masks[val_indices]

    train_image_data = {"image": train_images}
    train_mask_data = {"mask": train_masks}
    valid_image_data = {"image": val_images}
    valid_mask_data = {"mask": val_masks}

    ds_train_images = ds.NumpySlicesDataset(data=train_image_data, sampler=None, shuffle=False)
    ds_train_masks = ds.NumpySlicesDataset(data=train_mask_data, sampler=None, shuffle=False)

    if run_distribute:
        rank_id = get_rank()
        rank_size = get_group_size()
        ds_train_images = ds.NumpySlicesDataset(data=train_image_data,
                                                sampler=None,
                                                shuffle=False,
                                                num_shards=rank_size,
                                                shard_id=rank_id)
        ds_train_masks = ds.NumpySlicesDataset(data=train_mask_data,
                                               sampler=None,
                                               shuffle=False,
                                               num_shards=rank_size,
                                               shard_id=rank_id)

    ds_valid_images = ds.NumpySlicesDataset(data=valid_image_data, sampler=None, shuffle=False)
    ds_valid_masks = ds.NumpySlicesDataset(data=valid_mask_data, sampler=None, shuffle=False)

    c_resize_op = c_vision.Resize(size=(388, 388), interpolation=Inter.BILINEAR)
    c_pad = c_vision.Pad(padding=92)
    c_rescale_image = c_vision.Rescale(1.0/127.5, -1)
    c_rescale_mask = c_vision.Rescale(1.0/255.0, 0)

    c_trans_normalize_img = [c_rescale_image, c_resize_op, c_pad]
    c_trans_normalize_mask = [c_rescale_mask, c_resize_op, c_pad]
    c_center_crop = c_vision.CenterCrop(size=388)

    train_image_ds = ds_train_images.map(input_columns="image", operations=c_trans_normalize_img)
    train_mask_ds = ds_train_masks.map(input_columns="mask", operations=c_trans_normalize_mask)
    train_ds = ds.zip((train_image_ds, train_mask_ds))
    train_ds = train_ds.project(columns=["image", "mask"])
    if augment:
        augment_process = train_data_augmentation
        c_resize_op = c_vision.Resize(size=(572, 572), interpolation=Inter.BILINEAR)
        train_ds = train_ds.map(input_columns=["image", "mask"], operations=augment_process)
        train_ds = train_ds.map(input_columns="image", operations=c_resize_op)
        train_ds = train_ds.map(input_columns="mask", operations=c_resize_op)

    train_ds = train_ds.map(input_columns="mask", operations=c_center_crop)
    post_process = data_post_process
    train_ds = train_ds.map(input_columns=["image", "mask"], operations=post_process)
    train_ds = train_ds.shuffle(repeat*24)
    train_ds = train_ds.batch(batch_size=train_batch_size, drop_remainder=True)

    valid_image_ds = ds_valid_images.map(input_columns="image", operations=c_trans_normalize_img)
    valid_mask_ds = ds_valid_masks.map(input_columns="mask", operations=c_trans_normalize_mask)
    valid_ds = ds.zip((valid_image_ds, valid_mask_ds))
    valid_ds = valid_ds.project(columns=["image", "mask"])
    valid_ds = valid_ds.map(input_columns="mask", operations=c_center_crop)
    post_process = data_post_process
    valid_ds = valid_ds.map(input_columns=["image", "mask"], operations=post_process)
    valid_ds = valid_ds.batch(batch_size=1, drop_remainder=True)

    return train_ds, valid_ds

print("数据处理函数定义完成！")

## 四、定义训练回调函数

In [ ]:
class StepLossTimeMonitor(Callback):
    """训练监控回调函数"""
    def __init__(self, batch_size, per_print_times=1):
        super(StepLossTimeMonitor, self).__init__()
        if not isinstance(per_print_times, int) or per_print_times < 0:
            raise ValueError("print_step must be int and >= 0.")
        self._per_print_times = per_print_times
        self.batch_size = batch_size
        self.loss_history = []  # 记录损失历史

    def step_begin(self, run_context):
        self.step_time = time.time()

    def step_end(self, run_context):
        step_seconds = time.time() - self.step_time
        step_fps = self.batch_size*1.0/step_seconds

        cb_params = run_context.original_args()
        loss = cb_params.net_outputs

        if isinstance(loss, (tuple, list)):
            if isinstance(loss[0], Tensor) and isinstance(loss[0].asnumpy(), np.ndarray):
                loss = loss[0]

        if isinstance(loss, Tensor) and isinstance(loss.asnumpy(), np.ndarray):
            loss = np.mean(loss.asnumpy())

        cur_step_in_epoch = (cb_params.cur_step_num - 1) % cb_params.batch_num + 1

        if isinstance(loss, float) and (np.isnan(loss) or np.isinf(loss)):
            raise ValueError("epoch: {} step: {}. Invalid loss, terminating training.".format(
                cb_params.cur_epoch_num, cur_step_in_epoch))
        if self._per_print_times != 0 and cb_params.cur_step_num % self._per_print_times == 0:
            print("step: %s, loss is %s, fps is %s" % (cur_step_in_epoch, loss, step_fps), flush=True)
            self.loss_history.append(loss)


class LossHistory:
    """用于记录不同实验的损失历史"""
    def __init__(self):
        self.experiments = {}
    
    def add_experiment(self, name, losses):
        self.experiments[name] = losses
    
    def plot_comparison(self):
        """绘制不同实验的损失对比图"""
        plt.figure(figsize=(12, 6))
        for name, losses in self.experiments.items():
            plt.plot(losses, label=name, marker='o', markersize=3)
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.title('Training Loss Comparison')
        plt.legend()
        plt.grid(True)
        plt.show()

print("回调函数定义完成！")

## 五、定义训练函数

In [ ]:
def train_net(data_dir, cross_valid_ind=1, epochs=2, batch_size=16, lr=0.0001, 
              run_distribute=False, cfg=None, experiment_name="experiment"):
    """
    训练网络
    
    参数:
        data_dir: 数据目录
        cross_valid_ind: 交叉验证索引
        epochs: 训练轮数
        batch_size: 批次大小
        lr: 学习率
        cfg: 配置字典
        experiment_name: 实验名称
    
    返回:
        loss_monitor: 损失监控对象
    """
    if run_distribute:
        init()
        group_size = get_group_size()
        parallel_mode = ParallelMode.DATA_PARALLEL
        context.set_auto_parallel_context(parallel_mode=parallel_mode,
                                          device_num=group_size,
                                          gradients_mean=False)
    
    net = UNet(n_channels=cfg['num_channels'], n_classes=cfg['num_classes'])
    
    if cfg['resume']:
        param_dict = load_checkpoint(cfg['resume_ckpt'])
        load_param_into_net(net, param_dict)
    
    criterion = CrossEntropyWithLogits()
    train_dataset, _ = create_dataset(data_dir, epochs, batch_size, True, cross_valid_ind, run_distribute)
    train_data_size = train_dataset.get_dataset_size()
    print(f"[{experiment_name}] dataset length is: {train_data_size}")
    
    ckpt_config = CheckpointConfig(save_checkpoint_steps=train_data_size,
                                     keep_checkpoint_max=cfg['keep_checkpoint_max'])
    ckpt_dir = f'./ckpt_{experiment_name}/'
    os.makedirs(ckpt_dir, exist_ok=True)
    checkpoint_cb = ModelCheckpoint(prefix=f'ckpt_unet_{experiment_name}',
                                     directory=ckpt_dir,
                                     config=ckpt_config)
    
    optimizer = nn.Adam(params=net.trainable_params(),
                         learning_rate=lr,
                         weight_decay=cfg['weight_decay'],
                         loss_scale=cfg['loss_scale'])
    
    loss_scale_manager = mindspore.train.loss_scale_manager.FixedLossScaleManager(
        cfg['FixedLossScaleManager'], False)
    
    model = Model(net, loss_fn=criterion, loss_scale_manager=loss_scale_manager,
                 optimizer=optimizer, amp_level="O3")
    
    loss_monitor = StepLossTimeMonitor(batch_size=batch_size, per_print_times=20)
    
    print(f"[{experiment_name}] ======== Starting Training ========")
    print(f"[{experiment_name}] Parameters: lr={lr}, batch_size={batch_size}, epochs={epochs}")
    
    model.train(epochs, train_dataset, 
               callbacks=[loss_monitor, checkpoint_cb],
               dataset_sink_mode=False)
    
    print(f"[{experiment_name}] ======== End Training ========")
    
    return loss_monitor

print("训练函数定义完成！")

## 六、查看数据集

In [ ]:
# 加载并查看数据集
image = np.array([np.array(p) for p in ImageSequence.Iterator(Image.open("./data/train-volume.tif"))])
label = np.array([np.array(p) for p in ImageSequence.Iterator(Image.open("./data/train-labels.tif"))])

print(f"图像形状: {image.shape}")
print(f"标签形状: {label.shape}")

# 显示第一张图像和标签
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.imshow(image[0], cmap='gray')
plt.title('Original Image')
plt.subplot(2, 2, 2)
plt.imshow(label[0], cmap='gray')
plt.title('Ground Truth Label')
plt.show()

## 七、基准实验 - 原始参数

In [ ]:
# 基准实验配置
cfg_baseline = {
    'name': 'Unet',
    'lr': 0.0001,
    'epochs': 2,
    'distribute_epochs': 2,
    'batchsize': 16,
    'cross_valid_ind': 1,
    'num_classes': 2,
    'num_channels': 1,
    'keep_checkpoint_max': 10,
    'weight_decay': 0.0005,
    'loss_scale': 1024.0,
    'FixedLossScaleManager': 1024.0,
    'resume': False,
    'resume_ckpt': './',
}

print("=== 基准实验开始 ===")
print("参数设置:")
print(f"  学习率: {cfg_baseline['lr']}")
print(f"  批次大小: {cfg_baseline['batchsize']}")
print(f"  迭代轮数: {cfg_baseline['epochs']}")
print()

# 运行基准实验
data_url = './data'
run_distribute = False

loss_monitor_baseline = train_net(
    data_dir=data_url,
    cross_valid_ind=cfg_baseline['cross_valid_ind'],
    epochs=cfg_baseline['epochs'],
    batch_size=cfg_baseline['batchsize'],
    lr=cfg_baseline['lr'],
    run_distribute=run_distribute,
    cfg=cfg_baseline,
    experiment_name="baseline"
)

print("\n=== 基准实验完成 ===")

## 八、对比实验1 - 提高学习率

In [ ]:
# 对比实验1配置：提高学习率
cfg_high_lr = cfg_baseline.copy()
cfg_high_lr['lr'] = 0.001  # 学习率提高10倍

print("=== 对比实验1开始 ===")
print("参数设置:")
print(f"  学习率: {cfg_high_lr['lr']} (提高10倍)")
print(f"  批次大小: {cfg_high_lr['batchsize']}")
print(f"  迭代轮数: {cfg_high_lr['epochs']}")
print()

# 运行对比实验1
loss_monitor_high_lr = train_net(
    data_dir=data_url,
    cross_valid_ind=cfg_high_lr['cross_valid_ind'],
    epochs=cfg_high_lr['epochs'],
    batch_size=cfg_high_lr['batchsize'],
    lr=cfg_high_lr['lr'],
    run_distribute=run_distribute,
    cfg=cfg_high_lr,
    experiment_name="high_lr"
)

print("\n=== 对比实验1完成 ===")

## 九、对比实验2 - 减小批次大小

In [ ]:
# 对比实验2配置：减小批次大小
cfg_small_batch = cfg_baseline.copy()
cfg_small_batch['batchsize'] = 8  # 批次大小减半

print("=== 对比实验2开始 ===")
print("参数设置:")
print(f"  学习率: {cfg_small_batch['lr']}")
print(f"  批次大小: {cfg_small_batch['batchsize']} (减小)")
print(f"  迭代轮数: {cfg_small_batch['epochs']}")
print()

# 运行对比实验2
loss_monitor_small_batch = train_net(
    data_dir=data_url,
    cross_valid_ind=cfg_small_batch['cross_valid_ind'],
    epochs=cfg_small_batch['epochs'],
    batch_size=cfg_small_batch['batchsize'],
    lr=cfg_small_batch['lr'],
    run_distribute=run_distribute,
    cfg=cfg_small_batch,
    experiment_name="small_batch"
)

print("\n=== 对比实验2完成 ===")

## 十、实验结果对比分析

In [ ]:
# 创建损失历史对象
loss_history = LossHistory()

# 添加各实验的损失历史
loss_history.add_experiment(
    f"Baseline (lr={cfg_baseline['lr']}, batch={cfg_baseline['batchsize']})",
    loss_monitor_baseline.loss_history
)

loss_history.add_experiment(
    f"High LR (lr={cfg_high_lr['lr']}, batch={cfg_high_lr['batchsize']})",
    loss_monitor_high_lr.loss_history
)

loss_history.add_experiment(
    f"Small Batch (lr={cfg_small_batch['lr']}, batch={cfg_small_batch['batchsize']})",
    loss_monitor_small_batch.loss_history
)

# 绘制对比图
loss_history.plot_comparison()

# 打印统计信息
print("\n===== 实验结果统计 =====")
print(f"\n1. 基准实验 (lr={cfg_baseline['lr']}, batch={cfg_baseline['batchsize']}):")
print(f"   最终损失: {loss_monitor_baseline.loss_history[-1]:.6f}")
print(f"   平均损失: {np.mean(loss_monitor_baseline.loss_history):.6f}")
print(f"   最小损失: {np.min(loss_monitor_baseline.loss_history):.6f}")

print(f"\n2. 高学习率实验 (lr={cfg_high_lr['lr']}, batch={cfg_high_lr['batchsize']}):")
print(f"   最终损失: {loss_monitor_high_lr.loss_history[-1]:.6f}")
print(f"   平均损失: {np.mean(loss_monitor_high_lr.loss_history):.6f}")
print(f"   最小损失: {np.min(loss_monitor_high_lr.loss_history):.6f}")

print(f"\n3. 小批次实验 (lr={cfg_small_batch['lr']}, batch={cfg_small_batch['batchsize']}):")
print(f"   最终损失: {loss_monitor_small_batch.loss_history[-1]:.6f}")
print(f"   平均损失: {np.mean(loss_monitor_small_batch.loss_history):.6f}")
print(f"   最小损失: {np.min(loss_monitor_small_batch.loss_history):.6f}")

## 十一、实验分析总结

In [ ]:
# 实验分析总结
print("=" * 60)
print("实验二：Unet-5模型性能分析 - 总结")
print("=" * 60)

print("\n【实验目的】")
print("通过调节实验参数，分析不同参数设置对Unet模型性能的影响。")

print("\n【实验设计】")
print("本次实验对比了三种不同的参数设置：")
print("1. 基准实验: lr=0.0001, batch_size=16")
print("2. 高学习率实验: lr=0.001, batch_size=16")
print("3. 小批次实验: lr=0.0001, batch_size=8")

print("\n【参数影响分析】")
print("\n1. 学习率的影响:")
print("   - 学习率控制着模型参数更新的步长")
print("   - 较大的学习率(0.001)可能导致:")
print("     * 训练初期损失下降更快")
print("     * 可能导致训练不稳定，损失震荡")
print("     * 可能无法收敛到最优解")
print("   - 较小的学习率(0.0001)的特点:")
print("     * 训练更稳定")
print("     * 收敛速度较慢")
print("     * 通常能获得更好的最终性能")

print("\n2. 批次大小的影响:")
print("   - 批次大小影响梯度估计的准确性和训练速度")
print("   - 较大的批次(16)的特点:")
print("     * 梯度估计更准确，训练更稳定")
print("     * 内存占用更大")
print("     * 每个epoch的更新次数少")
print("   - 较小的批次(8)的特点:")
print("     * 每个epoch的更新次数更多")
print("     * 梯度估计有更多噪声")
print("     * 可能有助于跳出局部最优")
print("     * 训练速度可能更慢")

print("\n【实验结论】")
print("1. 学习率是影响模型训练的关键参数：")
print("   - 过大可能导致训练不稳定，无法收敛")
print("   - 过小会导致训练速度慢，计算资源浪费")
print("   - 需要根据具体任务和数据集进行调整")

print("2. 批次大小的选择需要权衡：")
print("   - 大批次: 训练稳定，但内存占用大")
print("   - 小批次: 更新频繁，但可能不稳定")
print("   - 在数据量较少时，较小的批次可能有助于泛化")

print("3. 参数调优建议：")
print("   - 建议先固定其他参数，单独调整学习率")
print("   - 可以使用学习率衰减策略，初期大后期小")
print("   - 批次大小应根据GPU内存和数据集大小选择")
print("   - 在实际应用中，建议进行网格搜索或随机搜索")

print("\n【进一步优化方向】")
print("1. 尝试不同的优化器(Adam, SGD, RMSprop等)")
print("2. 使用学习率调度器(Cosine, Step, Exponential等)")
print("3. 增加数据增强策略")
print("4. 调整网络结构(深度、宽度等)")
print("5. 使用正则化技术(Dropout, BatchNorm等)")

print("\n" + "=" * 60)
print("实验分析完成！")
print("=" * 60)

## 十二、模型验证

In [ ]:
def test_net(data_dir, ckpt_path, cross_valid_ind=1, cfg=None):
    """测试模型"""
    net = UNet(n_channels=cfg['num_channels'], n_classes=cfg['num_classes'])
    param_dict = load_checkpoint(ckpt_path)
    load_param_into_net(net, param_dict)
    
    criterion = CrossEntropyWithLogits()
    _, valid_dataset = create_dataset(data_dir, 1, 1, False, cross_valid_ind, False)
    
    model = Model(net, loss_fn=criterion, metrics={"dice_coeff": dice_coeff()})
    
    print("======== Starting Evaluating ========")
    dice_score = model.eval(valid_dataset, dataset_sink_mode=False)
    print("Cross valid dice coeff is:", dice_score)
    
    return dice_score

# 评估各实验的模型
print("\n===== 模型评估 =====")

# 评估基准模型
try:
    ckpt_path_baseline = './ckpt_baseline/ckpt_unet_baseline-2_{}.ckpt'.format(
        int(24 * 2 / cfg_baseline['batchsize'] * cfg_baseline['epochs']))
    if os.path.exists(ckpt_path_baseline):
        dice_baseline = test_net(data_url, ckpt_path_baseline, cfg_baseline['cross_valid_ind'], cfg_baseline)
        print(f"基准模型Dice系数: {dice_baseline}")
    else:
        print("基准模型checkpoint文件不存在")
except Exception as e:
    print(f"评估基准模型时出错: {e}")

# 评估高学习率模型
try:
    ckpt_path_high_lr = './ckpt_high_lr/ckpt_unet_high_lr-2_{}.ckpt'.format(
        int(24 * 2 / cfg_high_lr['batchsize'] * cfg_high_lr['epochs']))
    if os.path.exists(ckpt_path_high_lr):
        dice_high_lr = test_net(data_url, ckpt_path_high_lr, cfg_high_lr['cross_valid_ind'], cfg_high_lr)
        print(f"高学习率模型Dice系数: {dice_high_lr}")
    else:
        print("高学习率模型checkpoint文件不存在")
except Exception as e:
    print(f"评估高学习率模型时出错: {e}")

# 评估小批次模型
try:
    ckpt_path_small_batch = './ckpt_small_batch/ckpt_unet_small_batch-2_{}.ckpt'.format(
        int(24 * 2 / cfg_small_batch['batchsize'] * cfg_small_batch['epochs']))
    if os.path.exists(ckpt_path_small_batch):
        dice_small_batch = test_net(data_url, ckpt_path_small_batch, cfg_small_batch['cross_valid_ind'], cfg_small_batch)
        print(f"小批次模型Dice系数: {dice_small_batch}")
    else:
        print("小批次模型checkpoint文件不存在")
except Exception as e:
    print(f"评估小批次模型时出错: {e}")

print("\n===== 所有实验完成 ======")